# BioFuse Tutorial 3: Adding a New Foundation Model

This tutorial shows you how to integrate a new foundation model into BioFuse.

We'll cover:
1. Understanding BioFuse's model architecture
2. Adding a model to the configuration
3. Implementing the model loading logic
4. Testing the new model
5. Real-world examples

## Prerequisites

Your model should:
- Be a pre-trained vision model
- Output embeddings/features (not final predictions)
- Be accessible via HuggingFace, timm, or a custom checkpoint

## Part 1: Understanding BioFuse Model Architecture

BioFuse models are defined in two files:

1. **`biofuse/models/config.py`**: Defines `MODEL_MAP` with model metadata
2. **`biofuse/models/embedding_extractor.py`**: Implements `PreTrainedEmbedding` class

### Model Map Structure

```python
MODEL_MAP = {
    "ModelName": {
        "model": "path/to/model",      # HuggingFace ID, timm name, or checkpoint
        "tokenizer": None,              # Processor/tokenizer or custom transforms
        "timm_kwargs": {...}            # Optional: timm-specific arguments
    }
}
```

### PreTrainedEmbedding Class

The `PreTrainedEmbedding` class:
- Loads the model based on `MODEL_MAP`
- Freezes parameters (since we're extracting embeddings)
- Implements `forward()` to extract features

## Part 2: Adding a Model - Step by Step

Let's add a new model called **"MyVisionModel"** from HuggingFace.

### Step 1: Add to MODEL_MAP in `biofuse/models/config.py`

In [ ]:
# In biofuse/models/config.py

"""
MODEL_MAP = {
    # ... existing models ...
    
    "MyVisionModel": {
        "model": "organization/my-vision-model",  # HuggingFace model ID
        "tokenizer": None  # Will use AutoImageProcessor
    },
}
"""

print("Add this to MODEL_MAP in biofuse/models/config.py")

### Step 2: Add loading logic in `biofuse/models/embedding_extractor.py`

Add to the `_load_model()` method:

In [ ]:
# In biofuse/models/embedding_extractor.py, in the _load_model() method:

"""
def _load_model(self):
    model_info = MODEL_MAP.get(self.model_name)
    
    # ... existing model loading code ...
    
    elif self.model_name == "MyVisionModel":
        # Load from HuggingFace
        self.model = AutoModel.from_pretrained(
            model_info["model"],
            trust_remote_code=True  # If model uses custom code
        )
        self.processor = AutoImageProcessor.from_pretrained(
            model_info["model"]
        )
    
    else:
        raise ValueError(f"Unsupported model: {self.model_name}")
    
    # Move model to GPU
    self.model = self.model.to("cuda")
"""

print("Add this to _load_model() in PreTrainedEmbedding class")

### Step 3: Add forward pass logic

Add to the `forward()` method:

In [ ]:
# In biofuse/models/embedding_extractor.py, in the forward() method:

"""
def forward(self, input_data):
    with torch.no_grad():
        # ... existing model forward code ...
        
        elif self.model_name == "MyVisionModel":
            # Extract embeddings
            model_output = self.model(**input_data)
            # Get pooled output (usually the [CLS] token or mean pooling)
            outputs = model_output.pooler_output.clone()
            del model_output
        
        else:
            # Default: use last hidden state's first token
            outputs = self.model(input_data).last_hidden_state[:, 0, :]
    
    # Ensure correct shape
    if outputs.dim() == 1:
        outputs = outputs.unsqueeze(0)
    
    torch.cuda.empty_cache()
    return outputs
"""

print("Add this to forward() in PreTrainedEmbedding class")

### Step 4: Test the new model

In [ ]:
# Test the new model
from biofuse import BioFuse, load_medmnist

# Initialize with new model
biofuse = BioFuse(models=['MyVisionModel'], fusion_method='concat')

# Load a small dataset to test
train_data, num_classes = load_medmnist('pathmnist', split='train')

# Extract embeddings (test on small subset)
from torch.utils.data import Subset
small_subset = Subset(train_data, range(100))  # Only 100 samples

embeddings, labels = biofuse.extract_embeddings_from_loader(small_subset)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Success! Model is working.")

## Part 3: Different Model Types

BioFuse supports various model sources. Here are templates for each:

### A) HuggingFace Model with AutoModel

In [ ]:
# Example: DINOv2 from HuggingFace

# 1. Add to MODEL_MAP:
"""
"DINOv2": {
    "model": "facebook/dinov2-base",
    "tokenizer": None
}
"""

# 2. Load in _load_model():
"""
elif self.model_name == "DINOv2":
    self.model = AutoModel.from_pretrained(model_info["model"])
    self.processor = AutoImageProcessor.from_pretrained(model_info["model"])
"""

# 3. Forward pass:
"""
elif self.model_name == "DINOv2":
    model_output = self.model(**input_data)
    outputs = model_output.last_hidden_state[:, 0, :].clone()  # CLS token
    del model_output
"""

### B) timm Model (PyTorch Image Models)

In [ ]:
# Example: ResNet from timm

# 1. Add to MODEL_MAP:
"""
"ResNet50": {
    "model": "resnet50.a1_in1k",
    "tokenizer": transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
}
"""

# 2. Load in _load_model():
"""
elif self.model_name == "ResNet50":
    self.model = timm.create_model(
        model_info["model"],
        pretrained=True,
        num_classes=0  # Remove classification head, get features only
    )
    self.processor = model_info["tokenizer"]
"""

# 3. Forward pass:
"""
elif self.model_name == "ResNet50":
    model_output = self.model(input_data)
    outputs = model_output.clone()
    del model_output
"""

### C) OpenCLIP Model

In [ ]:
# Example: Custom CLIP model

# 1. Add to MODEL_MAP:
"""
"CustomCLIP": {
    "model": "hf-hub:org/custom-clip-model",
    "tokenizer": "hf-hub:org/custom-clip-model"
}
"""

# 2. Load in _load_model():
"""
elif self.model_name == "CustomCLIP":
    from open_clip import create_model_from_pretrained, get_tokenizer
    self.model, self.processor = create_model_from_pretrained(model_info["model"])
    self.tokenizer = get_tokenizer(model_info["tokenizer"])
"""

# 3. Forward pass:
"""
elif self.model_name == "CustomCLIP":
    outputs = self.model.encode_image(input_data)
"""

### D) Custom Checkpoint (Local File)

In [ ]:
# Example: Model with local checkpoint

# 1. Add to MODEL_MAP:
"""
"CustomModel": {
    "model": "vit_base_patch16_224",  # timm architecture
    "tokenizer": transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    "checkpoint_path": "/path/to/checkpoint.pth"
}
"""

# 2. Load in _load_model():
"""
elif self.model_name == "CustomModel":
    self.model = timm.create_model(
        model_info["model"],
        pretrained=False,
        num_classes=0
    )
    # Load custom checkpoint
    checkpoint = torch.load(model_info["checkpoint_path"], map_location="cpu")
    self.model.load_state_dict(checkpoint, strict=True)
    self.processor = model_info["tokenizer"]
"""

# 3. Forward pass:
"""
elif self.model_name == "CustomModel":
    model_output = self.model(input_data)
    outputs = model_output.clone()
    del model_output
"""

## Part 4: Real-World Example - Adding SigLIP

Let's walk through a complete example: adding Google's SigLIP model.

### Step 1: Add SigLIP to config.py

In [ ]:
# In biofuse/models/config.py

"""
MODEL_MAP = {
    # ... existing models ...
    
    "SigLIP": {
        "model": "google/siglip-base-patch16-224",
        "tokenizer": None
    },
}
"""

### Step 2: Add loading logic to embedding_extractor.py

In [ ]:
# In biofuse/models/embedding_extractor.py

# In _load_model():
"""
elif self.model_name == "SigLIP":
    from transformers import AutoModel, AutoImageProcessor
    self.model = AutoModel.from_pretrained(model_info["model"])
    self.processor = AutoImageProcessor.from_pretrained(model_info["model"])
"""

# In forward():
"""
elif self.model_name == "SigLIP":
    # SigLIP uses vision_model for image encoding
    model_output = self.model.vision_model(**input_data)
    outputs = model_output.pooler_output.clone()
    del model_output
"""

### Step 3: Test SigLIP

In [ ]:
# Test implementation
from biofuse import BioFuse, load_medmnist, get_classifier, compute_metrics

# Load dataset
train_data, num_classes = load_medmnist('pathmnist', split='train')
test_data, _ = load_medmnist('pathmnist', split='test')

# Initialize BioFuse with SigLIP
biofuse = BioFuse(models=['SigLIP'], fusion_method='concat')

# Extract embeddings
train_emb, train_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='train'
)

test_emb, test_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='test'
)

# Train classifier
clf = get_classifier('logistic', num_classes=num_classes)
clf.fit(train_emb, train_labels)

# Evaluate
pred = clf.predict(test_emb)
proba = clf.predict_proba(test_emb)
metrics = compute_metrics(test_labels, pred, proba, num_classes, task='multi-class')

print(f"SigLIP Accuracy: {metrics['accuracy']:.4f}")

### Step 4: Use in CLI and configs

In [ ]:
# CLI usage
!biofuse train --dataset pathmnist --models SigLIP --classifier xgboost

In [ ]:
# Config file (siglip_experiment.yaml)
"""
name: siglip_pathmnist
data:
  dataset: pathmnist
  img_size: 224
model:
  models: [SigLIP]
  fusion_method: concat
classifier:
  type: xgboost
"""

# Run with config
!biofuse train --config siglip_experiment.yaml

## Part 5: Troubleshooting

### Common Issues

**1. Shape Mismatch**
```python
# Problem: Output shape is wrong
# Solution: Check model output structure
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 224, 224).to('cuda')
    output = model(dummy_input)
    print(f"Output type: {type(output)}")
    print(f"Output shape: {output.shape if hasattr(output, 'shape') else 'N/A'}")
    if hasattr(output, '__dict__'):
        print(f"Available attributes: {list(output.__dict__.keys())}")
```

**2. Processor/Tokenizer Issues**
```python
# Some models need specific preprocessing
# Check model documentation for input format
```

**3. Memory Issues**
```python
# Always use .clone() and delete intermediate outputs
model_output = self.model(input_data)
outputs = model_output.pooler_output.clone()
del model_output  # Free memory
torch.cuda.empty_cache()
```

## Summary

To add a new model to BioFuse:

### Checklist

- [ ] Add model entry to `MODEL_MAP` in `biofuse/models/config.py`
- [ ] Add loading logic in `_load_model()` in `embedding_extractor.py`
- [ ] Add forward pass logic in `forward()` in `embedding_extractor.py`
- [ ] Test with a small dataset
- [ ] Test embedding shape is correct
- [ ] Test memory is freed properly
- [ ] Test with CLI
- [ ] Create example config file
- [ ] Document model requirements (GPU memory, licenses, etc.)

### Key Points

1. **Three files to modify**: `config.py`, `embedding_extractor.py`
2. **Always freeze parameters**: Models are used for feature extraction only
3. **Memory management**: Use `.clone()` and `del`, call `torch.cuda.empty_cache()`
4. **Output shape**: Ensure embeddings are 2D (batch_size, embedding_dim)
5. **Test thoroughly**: Verify on small dataset before full runs

## Next Tutorial

**Tutorial 4**: Learn how to add custom classifiers to BioFuse